# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [24]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [7]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [8]:
openai = OpenAI()

In [56]:
# Some lists!

checklist = []
completed = []
weights = []

In [57]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        weight = weights[index] if index < len(weights) else None
        weight_str = f" [azure](Weight: {weight}%)[/azure]" if weight is not None else ""
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]{weight_str}\n"
        else:
            result += f"Checklist #{index + 1}: {item}{weight_str}\n"
    show(result)
    return result

In [58]:
get_checklist_report()

''

In [59]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    weights.extend([0] * len(descriptions))
    return get_checklist_report()

In [60]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [61]:
def set_weightage(index: int, weight_percent: int) -> str:
    if 1 <= index <= len(checklist):
        weights[index - 1] = int(weight_percent)
    else:
        return "No checklist at this index."
    Console().print(f"Set weight for checklist #{index} to {weight_percent}%")
    return get_checklist_report()

In [62]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries (Weight: 0%)
Checklist #2: Finish week 1 (Weight: 0%)
Checklist #3: Eat banana (Weight: 0%)

'Checklist #1: Buy groceries [azure](Weight: 0%)[/azure]\nChecklist #2: Finish week 1 [azure](Weight: 0%)[/azure]\nChecklist #3: Eat banana [azure](Weight: 0%)[/azure]\n'

In [63]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries (Weight: 0%)
Checklist #2: Finish week 1 (Weight: 0%)
Checklist #3: Eat banana (Weight: 0%)

'Checklist #1: [green][strike]Buy groceries[/strike][/green] [azure](Weight: 0%)[/azure]\nChecklist #2: Finish week 1 [azure](Weight: 0%)[/azure]\nChecklist #3: Eat banana [azure](Weight: 0%)[/azure]\n'

In [64]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [65]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [66]:
weightage_json = {
    "name": "set_weightage",
    "description": "Set weight percentage for a checklist item at the given 1-based index and return the full list",
    "parameters": {
        "properties": {
            "index": {
                "description": "The 1-based index of the checklist item to set weight for",
                "title": "Index",
                "type": "integer"
            },
            "weight_percent": {
                "description": "Weight percentage to assign to the item (0-100)",
                "title": "Weight Percentage",
                "type": "integer"
            }
        },
        "required": ["index", "weight_percent"],
        "type": "object",
        "additionalProperties": False
    }
}

In [67]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json},
        {"type": "function", "function": weightage_json}]

In [68]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        print("tool call :", tool_call);
        tool_name = tool_call.function.name
        print("tool name :", tool_name);
        arguments = json.loads(tool_call.function.arguments)
        print("tool arguments :", arguments);
        tool = globals().get(tool_name)
        print("tool :", tool);
        result = tool(**arguments) if tool else {}
        print("content:", json.dumps(result))
        print("tool_call_id :", tool_call.id)
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [1]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB        

In [2]:
import requests
requests.get('http://localhost:11434').content

b'Ollama is running'

In [3]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model.get("id"))

llama3.2:latest
llama3.2:1b


In [4]:
ollama_model_name = "llama3.2:1b"


In [20]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [42]:
model_name_grok = "grok-4.3"
grok_api_key = "xai-QLPqZTfrVkI3U5oDCuVJ1mNBJwfjXPpL3cmY5YImZGlSzOZpW2dulqLv0wdVZJPuFzFsaWJYJATlIYsk"
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

GROK_BASE_URL = "https://api.x.ai/v1"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
# GROK_BASE_URL = "https://api.x.ai/v1"
grok = OpenAI(api_key=grok_api_key, base_url=GROK_BASE_URL)


# response = grok.chat.completions.create(model=model_name, messages=judge_messages)

In [43]:
groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)


In [69]:
model_openai = "openai/gpt-oss-120b"
model_openai_2 = "gpt-5.5"
def loop(messages):
    response = groq.chat.completions.create(model=model_openai, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = groq.chat.completions.create(model=model_openai, messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [70]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Please assign a specific weightage percentage to each step based on its complexity, effort, and importance, 
ensuring the total equals exactly 100%.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
"Analyze the net geopolitical influence gained by the United States through its post-WWII military interventions.
 Focus on how these invasions shifted global balances of power, the financial toll inflicted on the local economies of those countries, 
 and whether the long-term strategic benefits to US foreign policy ultimately outweighed the economic and diplomatic costs."
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [71]:
checklist, completed = [], []
loop(messages)

tool call : ChatCompletionMessageFunctionToolCall(id='fc_37fada39-9dea-4509-a64f-9d5c388cdf59', function=Function(arguments='{"descriptions":["Define scope: identify major post-WWII US military interventions to analyze","Collect data on geopolitical influence shifts resulting from each intervention","Assess financial impact on the local economies of the intervened countries","Evaluate long-term strategic benefits to US foreign policy from each intervention","Synthesize findings to determine net geopolitical influence versus costs","Draft comprehensive answer summarizing analysis with conclusions"]}', name='create_checklist'), type='function')
tool name : create_checklist
tool arguments : {'descriptions': ['Define scope: identify major post-WWII US military interventions to analyze', 'Collect data on geopolitical influence shifts resulting from each intervention', 'Assess financial impact on the local economies of the intervened countries', 'Evaluate long-term strategic benefits to US f

Checklist #1: Define scope: identify major post-WWII US military interventions to analyze (Weight: 0%)
Checklist #2: Collect data on geopolitical influence shifts resulting from each intervention (Weight: 0%)
Checklist #3: Assess financial impact on the local economies of the intervened countries (Weight: 0%)
Checklist #4: Evaluate long-term strategic benefits to US foreign policy from each intervention (Weight: 0%)
Checklist #5: Synthesize findings to determine net geopolitical influence versus costs (Weight: 0%)
Checklist #6: Draft comprehensive answer summarizing analysis with conclusions (Weight: 0%)

content: "Checklist #1: Define scope: identify major post-WWII US military interventions to analyze [azure](Weight: 0%)[/azure]\nChecklist #2: Collect data on geopolitical influence shifts resulting from each intervention [azure](Weight: 0%)[/azure]\nChecklist #3: Assess financial impact on the local economies of the intervened countries [azure](Weight: 0%)[/azure]\nChecklist #4: Evaluate long-term strategic benefits to US foreign policy from each intervention [azure](Weight: 0%)[/azure]\nChecklist #5: Synthesize findings to determine net geopolitical influence versus costs [azure](Weight: 0%)[/azure]\nChecklist #6: Draft comprehensive answer summarizing analysis with conclusions [azure](Weight: 0%)[/azure]\n"
tool_call_id : fc_37fada39-9dea-4509-a64f-9d5c388cdf59
tool call : ChatCompletionMessageFunctionToolCall(id='fc_7cbe15db-ff14-4371-9d96-7ebd9993a948', function=Function(arguments='{"index":1,"weight_percent":10}', name='set_weightage'), type='function')
tool name : set_weightage


Set weight for checklist #1 to 10%

Checklist #1: Define scope: identify major post-WWII US military interventions to analyze (Weight: 10%)
Checklist #2: Collect data on geopolitical influence shifts resulting from each intervention (Weight: 0%)
Checklist #3: Assess financial impact on the local economies of the intervened countries (Weight: 0%)
Checklist #4: Evaluate long-term strategic benefits to US foreign policy from each intervention (Weight: 0%)
Checklist #5: Synthesize findings to determine net geopolitical influence versus costs (Weight: 0%)
Checklist #6: Draft comprehensive answer summarizing analysis with conclusions (Weight: 0%)

...

content: "Checklist #1: [green][strike]Define scope: identify major post-WWII US military interventions to analyze[/strike][/green] [azure](Weight: 10%)[/azure]\nChecklist #2: [green][strike]Collect data on geopolitical influence shifts resulting from each intervention[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #3: [green][strike]Assess financial impact on the local economies of the intervened countries[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #4: [green][strike]Evaluate long-term strategic benefits to US foreign policy from each intervention[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #5: [green][strike]Synthesize findings to determine net geopolitical influence versus costs[/strike][/green] [azure](Weight: 15%)[/azure]\nChecklist #6: Draft comprehensive answer summarizing analysis with conclusions [azure](Weight: 15%)[/azure]\n"
tool_call_id : fc_659d2300-1682-4bdd-b89d-7f6cf04549b4
tool call : ChatCompletionMessageFunctionToolCall(id='fc_9a

**Net Geopolitical Influence of US Post‑WWII Military Interventions**

---
### 1. Scope of Analysis
The review focused on a representative set of major US‑led or US‑backed military actions after 1945 that have had 
lasting geopolitical ramifications:
- Korean War (1950‑53)
- Vietnam War (1965‑75)
- 1973 Chilean coup support
- Grenada invasion (1983)
- Panama invasion (1989)
- Gulf War (1990‑91)
- Somalia humanitarian/peacekeeping (1992‑93)
- Kosovo intervention (1999)
- Afghanistan War (2001‑2021)
- Iraq War (2003‑11)
- Libya intervention (2011)

---
### 2. Geopolitical Shifts
| Intervention | Key Geopolitical Outcome |
|---|---|
| **Korea** | Cemented US‑ROK alliance, permanent bases, deterrence against North Korea/China, solidified US role 
in East Asia. |
| **Vietnam** | Eroded US prestige, induced “Vietnam Syndrome”, but eventually led to détente and later strategic 
trade ties. |
| **Chile** | Secured a anti‑communist government, opened markets for US firms, generated regional backlash. |
| **Grenada** | Demonstrated rapid US response, deterred Soviet/Cuban influence in Caribbean. |
| **Panama** | Restored control over Canal, signaled intolerance for hostile regimes, later facilitated peaceful 
handover. |
| **Gulf War** | Re‑asserted US as the pre‑eminent post‑Cold‑War military power, built a broad coalition, deepened 
ties with Saudi Arabia and Israel. |
| **Somalia** | Highlighted limits of humanitarian intervention, led to more cautious future engagements. |
| **Kosovo** | Strengthened NATO’s role, prevented wider Balkan conflict, expanded US influence in Europe. |
| **Afghanistan** | Eliminated Al‑Qaeda safe haven, extended US reach into Central Asia, but nation‑building proved
costly. |
| **Iraq** | Removed Saddam, reshaped Middle‑East power balance (empowered Iran, sectarian fragmentation), opened 
oil contracts. |
| **Libya** | Enforced “Responsibility to Protect”, toppled Gaddafi, but created a power vacuum. |

---
### 3. Economic Toll on Target Nations
- **Combined direct war‑damage & reconstruction costs:** > $5 trillion (adjusted to 2023 USD).
- **Representative figures:**
  - Vietnam: $120‑150 bn (US) → 30 % GDP contraction.
  - Iraq: $2.4 trn (US) → ~20 % GDP loss, >$100 bn infrastructure destruction.
  - Afghanistan: $2.3 trn (US) → GDP remained < $20 bn, massive humanitarian deficit.
  - Gulf War (Kuwait/Iraq): > $200 bn combined losses.
- **Long‑term economic effects:** high inflation (Chile), debt dependence (Somalia, Kosovo), prolonged recession 
(Panama, Libya).

---
### 4. Long‑Term Strategic Benefits to the United States
- **Security Guarantees:** Permanent basing rights (Korea, Japan, Middle East), alliance networks (NATO, SEATO 
remnants).
- **Resource Access:** Secured maritime routes (Panama Canal), stable oil flow (Gulf, Iraq pre‑2003), contracts for
US firms.
- **Political Influence:** Ability to shape regional orders (East Asia, Middle East, Balkans), maintain global 
coalition leadership.
- **Doctrine Development:** Lessons from Vietnam, Somalia, Iraq informed later counter‑terrorism and 
nation‑building strategies.
- **Economic Gains:** Post‑conflict trade expansions (Vietnam, South Korea, Iraq oil contracts).

---
### 5. Net Assessment
| Category | Positive Outcomes | Negative Outcomes |
|---|---|---|
| **Geopolitical Influence** | Sustained US primacy, alliance expansion, control of strategic chokepoints. | 
Anti‑US sentiment, regional instability, rise of rival powers (e.g., Iran, China). |
| **Economic/Diplomatic Costs** | Access to markets & resources, long‑term trade partners. | Massive civilian 
suffering, reconstruction burdens, blow‑back (terrorism, insurgency). |
| **Strategic Balance** | Interventions that created lasting institutions (Korea, Panama, Gulf War) yielded a clear
net gain. | Interventions primarily aimed at regime change without solid post‑war frameworks (Vietnam, Iraq, 
Afghanistan, Libya) produced ambiguous or net‑negative returns. |

**Overall Conclusion**
The Unit

Checklist #1: Define scope: identify major post-WWII US military interventions to analyze (Weight: 10%)
Checklist #2: Collect data on geopolitical influence shifts resulting from each intervention (Weight: 20%)
Checklist #3: Assess financial impact on the local economies of the intervened countries (Weight: 20%)
Checklist #4: Evaluate long-term strategic benefits to US foreign policy from each intervention (Weight: 20%)
Checklist #5: Synthesize findings to determine net geopolitical influence versus costs (Weight: 15%)
Checklist #6: Draft comprehensive answer summarizing analysis with conclusions (Weight: 15%)

content: "Checklist #1: [green][strike]Define scope: identify major post-WWII US military interventions to analyze[/strike][/green] [azure](Weight: 10%)[/azure]\nChecklist #2: [green][strike]Collect data on geopolitical influence shifts resulting from each intervention[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #3: [green][strike]Assess financial impact on the local economies of the intervened countries[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #4: [green][strike]Evaluate long-term strategic benefits to US foreign policy from each intervention[/strike][/green] [azure](Weight: 20%)[/azure]\nChecklist #5: [green][strike]Synthesize findings to determine net geopolitical influence versus costs[/strike][/green] [azure](Weight: 15%)[/azure]\nChecklist #6: [green][strike]Draft comprehensive answer summarizing analysis with conclusions[/strike][/green] [azure](Weight: 15%)[/azure]\n"
tool_call_id : fc_9ae23191-8bea-4dc7-8576-4042a4d81a00


**Net Geopolitical Influence of United States Post‑WWII Military Interventions**  

---  

### 1. Scope of Analysis  
The review focused on a representative set of major US‑led or US‑backed actions after 1945 that have reshaped 
global power dynamics:  

- Korean War (1950‑53)  
- Vietnam War (1965‑75)  
- 1973 Chilean coup support  
- Grenada invasion (1983)  
- Panama invasion (1989)  
- Gulf War (1990‑91)  
- Somalia humanitarian/peacekeeping (1992‑93)  
- Kosovo intervention (1999)  
- Afghanistan War (2001‑2021)  
- Iraq War (2003‑11)  
- Libya intervention (2011)  

---  

### 2. Geopolitical Shifts  

| Intervention | Key Geopolitical Outcome |
|--------------|---------------------------|
| **Korea** | Cemented US‑ROK alliance, permanent bases, deterrence against North Korea/China; anchored US role in 
East Asia. |
| **Vietnam** | Eroded US prestige, produced “Vietnam Syndrome” that curtailed direct interventions; later 
normalized trade and strategic partnership. |
| **Chile** | Secured an anti‑communist regime, opened markets for US firms, but generated regional backlash and 
credibility concerns. |
| **Grenada** | Showed rapid US response capability, deterred Soviet/Cuban influence in the Caribbean. |
| **Panama** | Restored US control of the Canal, signaled intolerance for hostile regimes; later enabled peaceful 
transfer of the Canal (1999). |
| **Gulf War** | Re‑asserted US as the dominant post‑Cold‑War military power, forged a broad coalition, deepened 
ties with Saudi Arabia and Israel. |
| **Somalia** | Highlighted limits of humanitarian intervention; prompted more cautious future engagement 
doctrines. |
| **Kosovo** | Strengthened NATO’s role, prevented wider Balkan conflict, expanded US influence in Europe. |
| **Afghanistan** | Eliminated Al‑Qaeda’s safe haven, extended US reach into Central Asia, but nation‑building 
proved extremely costly. |
| **Iraq** | Removed Saddam, reshaped Middle‑East balance (empowered Iran, fragmented secular Arab nationalism), 
opened oil contracts for US firms; also created sectarian chaos. |
| **Libya** | Enforced “Responsibility to Protect”, toppled Gaddafi, but left a power vacuum and ongoing 
instability. |

---  

### 3. Economic Toll on Target Nations  

- **Combined direct war‑damage & reconstruction costs:** **> $5 trillion** (2023 USD).  
- **Representative figures** (US‑paid or total damage):  

  - **Vietnam:** $120‑150 bn (US); 30 % GDP contraction, devastated infrastructure.  
  - **Iraq:** $2.4 trn (US); ~20 % GDP loss, >$100 bn infrastructure destruction.  
  - **Afghanistan:** $2.3 trn (US); GDP remained < $20 bn, massive humanitarian deficit.  
  - **Gulf War (Kuwait/Iraq):** > $200 bn combined losses; oil‑revenue collapse.  
  - **Somalia, Kosovo, Panama, Grenada, Libya:** Each incurred billions in lost output, inflation spikes, and debt 
burdens.  

- **Long‑term effects:** high inflation (Chile), chronic debt (Somalia, Kosovo), prolonged recession (Panama, 
Libya), and persistent social disruption.  

---  

### 4. Long‑Term Strategic Benefits to the United States  

- **Security Guarantees:** Permanent basing rights (Korea, Japan, Middle East), enduring alliance networks (NATO, 
SEATO remnants).  
- **Resource Access:** Secure maritime routes (Panama Canal), stable oil flow (Gulf, pre‑2003 Iraq), lucrative 
contracts for US firms.  
- **Political Influence:** Ability to shape regional orders in East Asia, the Middle East, and the Balkans; 
maintain global coalition leadership.  
- **Doctrine Development:** Lessons from Vietnam, Somalia, Iraq refined US counter‑terrorism and nation‑building 
strategies.  
- **Economic Gains:** Post‑conflict trade expansion (Vietnam, South Korea), oil and reconstruction contracts (Iraq,
Afghanistan).  

---  

### 5. Net Assessment  

| Category | Positive Outcomes | Negative Outcomes |
|----------|-------------------|-------------------|
| **Geopolitical Influence** | Sustained US primacy, expanded alliances, control of stra

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>